In [1]:
# Install required libraries
# !pip install yfinance pandas numpy scipy curl_cffi requests

"""
퀀트 투자를 위한 주식 데이터 수집 및 기술적 지표 계산
"""

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from scipy.stats import linregress
import time
import random
from typing import List, Dict, Optional, Tuple, Union
import requests

# 브라우저 모방을 위한 세션 설정
try:
    from curl_cffi import requests as curl_requests
    try:
        # Chrome 120 버전을 모방하는 세션 생성
        yf_session = curl_requests.Session(impersonate="chrome120")
        yf_session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        })
    except Exception as e_session:
        yf_session = requests.Session()
        yf_session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        })
except ImportError:
    yf_session = requests.Session()
    yf_session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    })

from sklearn.preprocessing import MinMaxScaler

def scale_features(df: pd.DataFrame, exclude_columns: List[str] = ['Date', 'Close']) -> pd.DataFrame:
    df_scaled = df.copy()
    feature_columns = [col for col in df.columns if col not in exclude_columns]
    
    scaler = MinMaxScaler()
    df_scaled[feature_columns] = scaler.fit_transform(df[feature_columns])
    
    return df_scaled

def calculate_technical_indicators(data: pd.DataFrame) -> pd.DataFrame:
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    data['RSI'] = 100 - (100 / (1 + rs))

    exp1 = data['Close'].ewm(span=12, adjust=False).mean()
    exp2 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = exp1 - exp2
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['MACD_Signal']

    data['BB_Middle'] = data['Close'].rolling(window=20).mean()
    data['BB_Std'] = data['Close'].rolling(window=20).std()
    data['BB_Upper'] = data['BB_Middle'] + (data['BB_Std'] * 2)
    data['BB_Lower'] = data['BB_Middle'] - (data['BB_Std'] * 2)

    high_low = data['High'] - data['Low']
    high_close = np.abs(data['High'] - data['Close'].shift())
    low_close = np.abs(data['Low'] - data['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    data['ATR'] = true_range.rolling(14).mean()

    tp = (data['High'] + data['Low'] + data['Close']) / 3
    data['CCI'] = (tp - tp.rolling(20).mean()) / (0.015 * tp.rolling(20).std())

    typical_price = (data['High'] + data['Low'] + data['Close']) / 3
    money_flow = (typical_price * data['Volume']).astype(float)

    positive_flow = pd.Series(0.0, index=data.index)
    negative_flow = pd.Series(0.0, index=data.index)
    positive_flow[typical_price > typical_price.shift(1)] = money_flow[typical_price > typical_price.shift(1)]
    negative_flow[typical_price < typical_price.shift(1)] = money_flow[typical_price < typical_price.shift(1)]

    money_ratio = positive_flow.rolling(14).sum() / (negative_flow.rolling(14).sum() + 1e-9)
    mfi = 100 - (100 / (1 + money_ratio))
    data['MFI'] = mfi

    # 🔍 하나라도 NaN이 있는 행 제거 및 해당 날짜 출력
    any_nan_rows = data[data.isnull().any(axis=1)]
    if not any_nan_rows.empty:
        print("⚠️ 하나 이상의 열이 비어있는 날짜:")
        if 'Date' in data.columns:
            print(data.loc[any_nan_rows.index, 'Date'].tolist())
        else:
            print(any_nan_rows.index.tolist())
        data = data.drop(index=any_nan_rows.index)

    return data

def calculate_price_patterns(data: pd.DataFrame) -> pd.DataFrame:
    """가격 패턴 계산"""
    # 이전 고점/저점
    data['Previous_High'] = data['High'].rolling(window=20).max()
    data['Previous_Low'] = data['Low'].rolling(window=20).min()
    
    # 추세선 (20일 기준)
    data['Trend_20'] = data['Close'].rolling(window=20).mean()
    
    # 가격 변동성
    data['Price_Volatility'] = data['Close'].rolling(window=20).std() / data['Close'].rolling(window=20).mean()
    
    # 모멘텀
    data['Momentum'] = data['Close'] - data['Close'].shift(10)
    
    # 가격 범위
    data['Price_Range'] = (data['High'] - data['Low']) / data['Close']
    
    return data

def calculate_volume_profile(data: pd.DataFrame) -> pd.DataFrame:
    """거래량 프로파일 계산"""
    # 거래량 이동평균
    data['Volume_MA5'] = data['Volume'].rolling(window=5).mean()
    data['Volume_MA20'] = data['Volume'].rolling(window=20).mean()
    
    # VWAP (Volume Weighted Average Price)
    data['VWAP'] = (data['Volume'] * (data['High'] + data['Low'] + data['Close']) / 3).cumsum() / data['Volume'].cumsum()
    
    # 거래량 변동성
    data['Volume_Volatility'] = data['Volume'].rolling(window=20).std() / data['Volume'].rolling(window=20).mean()
    
    # OBV (On-Balance Volume)
    data['OBV'] = (np.sign(data['Close'].diff()) * data['Volume']).fillna(0).cumsum()
    
    # 거래량 가중 가격
    data['Volume_Weighted_Price'] = (data['Volume'] * data['Close']) / data['Volume']
    
    return data

def calculate_adx(data: pd.DataFrame, period: int = 14) -> pd.Series:
    plus_dm = data['High'].diff()
    minus_dm = data['Low'].diff().abs()
    tr1 = data['High'] - data['Low']
    tr2 = abs(data['High'] - data['Close'].shift())
    tr3 = abs(data['Low'] - data['Close'].shift())
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(period).mean()

    plus_di = 100 * (plus_dm.rolling(period).mean() / atr)
    minus_di = 100 * (minus_dm.rolling(period).mean() / atr)
    dx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(period).mean()
    return adx


class MovingAverageCalculator:
    """이동평균 및 기울기 계산을 위한 클래스"""
    
    @staticmethod
    def calculate_ma(data: pd.Series, window: int) -> pd.Series:
        """이동 평균을 계산"""
        return data.rolling(window=window, min_periods=1).mean()
    
    @staticmethod
    def calculate_slope(data: pd.Series) -> pd.Series:
        """이동 평균선의 기울기(전일 대비 변화량)를 계산"""
        return data.diff() / 2

class StockDataProcessor:
    """주식 데이터 처리 및 저장을 위한 클래스"""
    
    def __init__(self, project_dir: str = 'cursor_csv'):
        """초기화"""
        self.project_dir = project_dir
        os.makedirs(project_dir, exist_ok=True)
        self.ma_calculator = MovingAverageCalculator()
    
    def get_stock_data(self, 
                    ticker: str, 
                    start_date: str, 
                    end_date: str, 
                    interval: str = '1d',
                    ma_periods: Optional[List[int]] = None) -> pd.DataFrame:
        """주식 데이터를 다운로드하고 이동평균 및 기울기 계산"""
        # 기본 이동평균 기간 설정
        if ma_periods is None:
            ma_periods = [5, 10, 15, 20, 25, 30, 60, 90, 120, 240, 480]
        
        # 간격에 따른 접두사 설정
        prefix = {
            '1d': 'D',   # 일봉
            '1wk': 'W',  # 주봉
            '1mo': 'M'   # 월봉
        }.get(interval, '')
        
        # 데이터 다운로드
        try:
            data = yf.download(
                ticker,
                start=start_date,
                end=end_date,
                interval=interval,
                auto_adjust=False,
                timeout=30,
                session=yf_session  # 세션 추가
            )
            
            if data.empty:
                print(f"❌ 데이터를 가져오지 못했습니다. 티커: {ticker}, 기간: {start_date} ~ {end_date}, 간격: {interval}")
                return pd.DataFrame()
                
        except Exception as e:
            print(f"⚠️ 오류 발생: {ticker}, 오류: {str(e)}")
            return pd.DataFrame()
        
        # 필요한 컬럼만 선택하고 인덱스 재설정
        data = data[['Open', 'High', 'Low', 'Close', 'Volume']].reset_index()
        data.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
        
        # 데이터 타입 변환
        data['Volume'] = data['Volume'].astype('float64') 

        # 기술적 지표 계산
        data = calculate_technical_indicators(data)

        # ✅ 사용자 정의 지표 추가
        data['ADX'] = calculate_adx(data)
        data['ZScore_Close'] = (data['Close'] - data['Close'].rolling(20).mean()) / data['Close'].rolling(20).std()
        data['Price_Shock'] = data['Close'].pct_change().apply(lambda x: 1 if x > 0.1 else (-1 if x < -0.1 else 0))
        
        # 가격 패턴 계산
        data = calculate_price_patterns(data)
        
        # 거래량 프로파일 계산
        data = calculate_volume_profile(data)
        
        # 이동평균 및 기울기 계산
        ma_columns = {}
        slope_columns = {}
        
        for period in ma_periods:
            # 가격 이동평균
            ma_price = self.ma_calculator.calculate_ma(data['Close'], period)
            ma_columns[f'SMA_{period}'] = ma_price
            slope_columns[f'Slope_SMA_{period}'] = self.ma_calculator.calculate_slope(ma_price)
            
            # 거래량 이동평균
            ma_volume = self.ma_calculator.calculate_ma(data['Volume'], period)
            ma_columns[f'VMA_{period}'] = ma_volume
            slope_columns[f'Slope_VMA_{period}'] = self.ma_calculator.calculate_slope(ma_volume)
        
        # 데이터프레임 생성 및 병합
        ma_df = pd.DataFrame(ma_columns)
        slope_df = pd.DataFrame(slope_columns)
        result_df = pd.concat([data, ma_df, slope_df], axis=1)
        
        # NaN 값 제거
        result_df.dropna(inplace=True)

        # 인덱스 재설정
        result_df.reset_index(drop=True, inplace=True)

        # ✅ Close 컬럼을 0번째로 이동
        cols = result_df.columns.tolist()
        if 'Close' in cols:
            cols.insert(0, cols.pop(cols.index('Close')))
            result_df = result_df[cols]
        
        return result_df
    
    def merge_interval_data(self, daily_data: pd.DataFrame, weekly_data: pd.DataFrame, monthly_data: pd.DataFrame) -> pd.DataFrame:
        """일별/주별/월별 데이터를 병합"""

        result = daily_data.copy()

        if not weekly_data.empty:
            # 날짜 인덱스 설정 후 일별로 리샘플링
            weekly_data = weekly_data.set_index('Date').resample('D').ffill().reset_index()
            weekly_data = weekly_data.add_prefix('W_')
            weekly_data.rename(columns={'W_Date': 'Date'}, inplace=True)

            result = pd.merge(result, weekly_data, on='Date', how='left')

        if not monthly_data.empty:
            # 날짜 인덱스 설정 후 일별로 리샘플링
            monthly_data = monthly_data.set_index('Date').resample('D').ffill().reset_index()
            monthly_data = monthly_data.add_prefix('M_')
            monthly_data.rename(columns={'M_Date': 'Date'}, inplace=True)

            result = pd.merge(result, monthly_data, on='Date', how='left')

        # 혹시 남은 NaN이 있으면 forward fill
        result.fillna(method='ffill', inplace=True)

        # 혹시 남은 NaN이 있으면 제거 및 로그 출력
        any_nan_rows = result[result.isnull().any(axis=1)]
        if not any_nan_rows.empty:
            print("⚠️ 병합 후 하나라도 NaN인 날짜들:")
            print(result.loc[any_nan_rows.index, 'Date'].tolist())
            result = result.drop(index=any_nan_rows.index)

        return result
    
    def save_data(self, data: pd.DataFrame, filename: str) -> None:
        """데이터를 CSV 파일로 저장"""
        if data.empty:
            print(f"❌ 저장할 데이터가 없습니다: {filename}")
            return
            
        # ✅ 스케일링
        scaled_data = scale_features(data)

        # 저장
        full_path = os.path.join(self.project_dir, filename)
        scaled_data.to_csv(full_path, index=False)
        print(f"✅ 스케일링 후 저장되었습니다: {full_path} (행 수: {len(scaled_data)})")

class StockDataManager:
    """주식 데이터 관리 클래스"""
    
    def __init__(self, project_dir: str = 'cursor_csv'):
        """초기화"""
        self.processor = StockDataProcessor(project_dir)
    
    def process_ticker(self, 
                      ticker: str, 
                      start_date: str, 
                      end_date: str,
                      train_test_split_date: Optional[str] = None) -> None:
        """특정 티커의 데이터를 처리하고 저장"""
        ticker_name = ticker.replace('^', '')  # 특수문자 제거
        
        if train_test_split_date:
            # 학습 데이터
            print(f"📥 {ticker} 학습 데이터 다운로드 중...")
            
            # 일별/주별/월별 데이터 수집
            daily_train = self.processor.get_stock_data(ticker, start_date, train_test_split_date, interval='1d')
            weekly_train = self.processor.get_stock_data(ticker, start_date, train_test_split_date, interval='1wk')
            monthly_train = self.processor.get_stock_data(ticker, start_date, train_test_split_date, interval='1mo')
            
            # 데이터 병합
            combined_train = self.processor.merge_interval_data(daily_train, weekly_train, monthly_train)
            
            if not combined_train.empty:
                self.processor.save_data(combined_train, f"{ticker_name}_train_data.csv")
            
            # 테스트 데이터
            print(f"📥 {ticker} 테스트 데이터 다운로드 중...")
            
            # 일별/주별/월별 데이터 수집
            daily_test = self.processor.get_stock_data(ticker, train_test_split_date, end_date, interval='1d')
            weekly_test = self.processor.get_stock_data(ticker, train_test_split_date, end_date, interval='1wk')
            monthly_test = self.processor.get_stock_data(ticker, train_test_split_date, end_date, interval='1mo')
            
            # 데이터 병합
            combined_test = self.processor.merge_interval_data(daily_test, weekly_test, monthly_test)
            
            if not combined_test.empty:
                self.processor.save_data(combined_test, f"{ticker_name}_test_data.csv")
        else:
            # 전체 데이터
            print(f"📥 {ticker} 데이터 다운로드 중...")
            
            # 일별/주별/월별 데이터 수집
            daily_data = self.processor.get_stock_data(ticker, start_date, end_date, interval='1d')
            weekly_data = self.processor.get_stock_data(ticker, start_date, end_date, interval='1wk')
            monthly_data = self.processor.get_stock_data(ticker, start_date, end_date, interval='1mo')
            
            # 데이터 병합
            combined_data = self.processor.merge_interval_data(daily_data, weekly_data, monthly_data)
            
            if not combined_data.empty:
                self.processor.save_data(combined_data, f"{ticker_name}_data.csv")
    
    def process_multiple_tickers(self, 
                               tickers: List[str], 
                               start_date: str, 
                               end_date: str,
                               train_test_split_date: Optional[str] = None) -> None:
        """여러 티커에 대해 데이터를 처리"""
        failed_tickers = []
        
        for ticker in tickers:
            print(f"\n🔍 처리 중: {ticker}")
            try:
                self.process_ticker(ticker, start_date, end_date, train_test_split_date)
                # 요청 제한을 피하기 위한 대기 시간
                time.sleep(random.uniform(2, 4))
            except Exception as e:
                print(f"❌ {ticker} 처리 중 오류 발생: {str(e)}")
                failed_tickers.append(ticker)
                continue
        
        if failed_tickers:
            print(f"\n⚠️ 실패한 티커: {', '.join(failed_tickers)}")
            print("🔄 실패한 티커 재시도 중...")
            for ticker in failed_tickers:
                print(f"\n🔍 재시도 중: {ticker}")
                try:
                    self.process_ticker(ticker, start_date, end_date, train_test_split_date)
                    time.sleep(random.uniform(3, 5))
                except Exception as e:
                    print(f"❌ {ticker} 재시도 실패: {str(e)}")

def main():
    """메인 함수"""
    # 설정
    project_dir = 'cursor_csv'
    start_date = '2006-01-01'
    train_test_split_date = '2023-03-01'
    end_date = '2025-02-12'
    
    # 관리자 인스턴스 생성
    manager = StockDataManager(project_dir)
    
    # 📌 저장할 주식 리스트 (아마존만)
    tickers = ['AMZN']  # 아마존
    
    # 데이터 처리
    print(f"\n📊 {len(tickers)}개 종목 데이터 처리 시작...")
    manager.process_multiple_tickers(tickers, start_date, end_date, train_test_split_date)
    print("\n✨ 모든 데이터 처리 완료!")

if __name__ == "__main__":
    main()


📊 1개 종목 데이터 처리 시작...

🔍 처리 중: AMZN
📥 AMZN 학습 데이터 다운로드 중...


[*********************100%***********************]  1 of 1 completed


⚠️ 하나 이상의 열이 비어있는 날짜:
[Timestamp('2006-01-03 00:00:00'), Timestamp('2006-01-04 00:00:00'), Timestamp('2006-01-05 00:00:00'), Timestamp('2006-01-06 00:00:00'), Timestamp('2006-01-09 00:00:00'), Timestamp('2006-01-10 00:00:00'), Timestamp('2006-01-11 00:00:00'), Timestamp('2006-01-12 00:00:00'), Timestamp('2006-01-13 00:00:00'), Timestamp('2006-01-17 00:00:00'), Timestamp('2006-01-18 00:00:00'), Timestamp('2006-01-19 00:00:00'), Timestamp('2006-01-20 00:00:00'), Timestamp('2006-01-23 00:00:00'), Timestamp('2006-01-24 00:00:00'), Timestamp('2006-01-25 00:00:00'), Timestamp('2006-01-26 00:00:00'), Timestamp('2006-01-27 00:00:00'), Timestamp('2006-01-30 00:00:00')]


[*********************100%***********************]  1 of 1 completed


⚠️ 하나 이상의 열이 비어있는 날짜:
[Timestamp('2006-01-01 00:00:00'), Timestamp('2006-01-08 00:00:00'), Timestamp('2006-01-15 00:00:00'), Timestamp('2006-01-22 00:00:00'), Timestamp('2006-01-29 00:00:00'), Timestamp('2006-02-05 00:00:00'), Timestamp('2006-02-12 00:00:00'), Timestamp('2006-02-19 00:00:00'), Timestamp('2006-02-26 00:00:00'), Timestamp('2006-03-05 00:00:00'), Timestamp('2006-03-12 00:00:00'), Timestamp('2006-03-19 00:00:00'), Timestamp('2006-03-26 00:00:00'), Timestamp('2006-04-02 00:00:00'), Timestamp('2006-04-09 00:00:00'), Timestamp('2006-04-16 00:00:00'), Timestamp('2006-04-23 00:00:00'), Timestamp('2006-04-30 00:00:00'), Timestamp('2006-05-07 00:00:00')]


...

⚠️ 하나 이상의 열이 비어있는 날짜:
[Timestamp('2023-03-01 00:00:00'), Timestamp('2023-03-02 00:00:00'), Timestamp('2023-03-03 00:00:00'), Timestamp('2023-03-06 00:00:00'), Timestamp('2023-03-07 00:00:00'), Timestamp('2023-03-08 00:00:00'), Timestamp('2023-03-09 00:00:00'), Timestamp('2023-03-10 00:00:00'), Timestamp('2023-03-13 00:00:00'), Timestamp('2023-03-14 00:00:00'), Timestamp('2023-03-15 00:00:00'), Timestamp('2023-03-16 00:00:00'), Timestamp('2023-03-17 00:00:00'), Timestamp('2023-03-20 00:00:00'), Timestamp('2023-03-21 00:00:00'), Timestamp('2023-03-22 00:00:00'), Timestamp('2023-03-23 00:00:00'), Timestamp('2023-03-24 00:00:00'), Timestamp('2023-03-27 00:00:00')]


[*********************100%***********************]  1 of 1 completed


⚠️ 하나 이상의 열이 비어있는 날짜:
[Timestamp('2023-02-27 00:00:00'), Timestamp('2023-03-06 00:00:00'), Timestamp('2023-03-13 00:00:00'), Timestamp('2023-03-20 00:00:00'), Timestamp('2023-03-27 00:00:00'), Timestamp('2023-04-03 00:00:00'), Timestamp('2023-04-10 00:00:00'), Timestamp('2023-04-17 00:00:00'), Timestamp('2023-04-24 00:00:00'), Timestamp('2023-05-01 00:00:00'), Timestamp('2023-05-08 00:00:00'), Timestamp('2023-05-15 00:00:00'), Timestamp('2023-05-22 00:00:00'), Timestamp('2023-05-29 00:00:00'), Timestamp('2023-06-05 00:00:00'), Timestamp('2023-06-12 00:00:00'), Timestamp('2023-06-19 00:00:00'), Timestamp('2023-06-26 00:00:00'), Timestamp('2023-07-03 00:00:00')]


[*********************100%***********************]  1 of 1 completed
C:\Users\kkkyg\AppData\Local\Temp\ipykernel_6308\268773455.py:295: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  result.fillna(method='ffill', inplace=True)


⚠️ 하나 이상의 열이 비어있는 날짜:
[Timestamp('2023-03-01 00:00:00'), Timestamp('2023-04-01 00:00:00'), Timestamp('2023-05-01 00:00:00'), Timestamp('2023-06-01 00:00:00'), Timestamp('2023-07-01 00:00:00'), Timestamp('2023-08-01 00:00:00'), Timestamp('2023-09-01 00:00:00'), Timestamp('2023-10-01 00:00:00'), Timestamp('2023-11-01 00:00:00'), Timestamp('2023-12-01 00:00:00'), Timestamp('2024-01-01 00:00:00'), Timestamp('2024-02-01 00:00:00'), Timestamp('2024-03-01 00:00:00'), Timestamp('2024-04-01 00:00:00'), Timestamp('2024-05-01 00:00:00'), Timestamp('2024-06-01 00:00:00'), Timestamp('2024-07-01 00:00:00'), Timestamp('2024-08-01 00:00:00'), Timestamp('2024-09-01 00:00:00')]
⚠️ 병합 후 하나라도 NaN인 날짜들:
[Timestamp('2023-05-05 00:00:00'), Timestamp('2023-05-08 00:00:00'), Timestamp('2023-05-09 00:00:00'), Timestamp('2023-05-10 00:00:00'), Timestamp('2023-05-11 00:00:00'), Timestamp('2023-05-12 00:00:00'), Timestamp('2023-05-15 00:00:00'), Timestamp('2023-05-16 00:00:00'), Timestamp('2023-05-17 00:00:00'), T